# Real-data figures

Every real-data figure in the paper, from the saved results. `DATASET` selects embryoid or
statefate; `SHIPPED = 0` switches the reads from the shipped paper results to your own
`new_results/` re-run.

1. **Estimated distributions** -- every cloud of the anchored series in PC1/PC2, coloured by time.
2. **Trajectory comparison** -- the kept cells' paths, one panel per method, metrics beside it.
3. **The fate clustering** -- the same paths coloured by cluster.
4. **Gene dynamics** -- how each fate's marker gene moves along the estimated time grid.

In [ ]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch
import matplotlib.patheffects as path_effects


_here = os.path.abspath(os.path.dirname(__file__)) if "__file__" in globals() else os.getcwd()
# this folder (its helper modules) + tools/ (the shared `_repo.py` path resolver)
for _d in (_here, os.path.abspath(os.path.join(_here, os.pardir, os.pardir, "tools"))):
    if _d not in sys.path:
        sys.path.insert(0, _d)
import _analysis_common as A                               # shared loaders (also puts `src/` on the path)
import _repo as P
import _traj_paths as TP                                   # per-method folders (ours/WOT/MMFM/MioFlow/TIGON)
from _traj_paths import FIGURE_METHODS                     # the five panels, in order

REPO = A.ROOT

In [ ]:
DATASET = globals().get("DATASET", "embryoid")    # "embryoid" | "statefate" -- run once each
DIM     = globals().get("DIM", 20)

_DS = {"embryoid":  dict(seed="5", n_per_time=800, n_bg=500, skip_pending=0),
       "statefate": dict(seed="2", n_per_time=600, n_bg=400, skip_pending=1)}[DATASET]
SEED    = globals().get("SEED", _DS["seed"])   # the seed of record per dataset
SHIPPED = globals().get("SHIPPED", 1)   # 1 = read the shipped paper results; 0 = your new_results/ run

# ---- how many cells to DRAW (never all of them) -------------------------------------------------
N_PER_TIME = globals().get("N_PER_TIME", _DS["n_per_time"])   # §1: cells scattered per time point
N_BG       = globals().get("N_BG", _DS["n_bg"])               # §2: background cells per time point
N_TRAJ     = 20   # §2: trajectories drawn per panel
BG_SEED    = 42                                        # which cells the subsample picks (reproducible)

# ---- §2 background style ------------------------------------------------------------------------
BG_MODE   = "grey"          # "grey" (light grey, best for reading paths) | "time" (RdBu_r by time)
BG_SOURCE = "observed"      # "observed" = the raw measured cells | "series" = the anchored series
BG_GREY   = "0.80"          # the grey; lower = darker
BG_ALPHA  = 0.55
BG_S      = 8               # background marker size

# ---- §2 path style ------------------------------------------------------------------------------
PATH_LW    = 0.95           # path line width
PATH_MS    = 7              # arrowhead size
PATH_ARROWS = 2             # arrowheads per path, spread along it (the last segment always gets one)
PATH_ALPHA = 0.85           # path opacity
PATH_HALO  = 0.0            # white casing width under the line, in points (0 = off; set per style)
NODE_S     = 22             # waypoint marker size
ENDNODE_S  = 38             # first/last waypoint marker size
NODE_EDGE  = 0.55           # waypoint outline width
NODE_EDGE_MID = "0.35"      # rim colour of the INTERIOR waypoints (endpoints keep a black rim)

# ---- §2 styles: where the time information sits -------------------------------------------------
#   "arrows"  time read off a faint time-coloured background; paths are arrows, no waypoints
#   "dots"    time read off the path's own coloured waypoints; background plain grey
# Every §2 drawing call takes `style=` to override the default.
STYLES = {
    "arrows": dict(bg_mode="time", bg_alpha=0.28, bg_n=300, nodes="none",  node_scale=1.0,
                   n_arrows=2, halo=0.9),
    "dots":   dict(bg_mode="grey", bg_alpha=0.40, bg_n=300, nodes="small", node_scale=0.45,
                   n_arrows=1, halo=0.0),
}
STYLE = "arrows"         # the paper figure


def _style(name=None):
    st = STYLES[name or STYLE].copy()
    return st

# ---- time colours: the paper's palette; everything (clouds, waypoints, legends) follows ---------
CMAP_NAME  = "RdBu_r"
CMAP_RANGE = (0.125, 0.875)   # the slice used, so the end days avoid RdBu_r's near-black extremes

# text size: printed pt = fontsize x 6.5 / figsize-width (the figure goes in at `width=\linewidth`)
FS = globals().get("FS", 1.30)
# 1 = drop a method with no saved run; 0 = keep its panel, labelled "run pending"
SKIP_PENDING = globals().get("SKIP_PENDING", _DS["skip_pending"])
plt.rcParams.update({"font.size": 11 * FS, "axes.titlesize": 12 * FS, "axes.labelsize": 11 * FS,
                     "xtick.labelsize": 10 * FS, "ytick.labelsize": 10 * FS, "legend.fontsize": 9 * FS,
                     "figure.titlesize": 15 * FS})   # 13 -> 15: the suptitle reads small once the
#                                                     wide canvas is rescaled to the text block

In [ ]:
RESULTS = A.results_dir(DATASET, DIM, shipped=SHIPPED)     # .../<ds>/d<DIM>
TUNE    = os.path.join(os.path.dirname(RESULTS), f"d{DIM}_tune")
SEEDDIR = RESULTS if str(SEED) in ("main", "") else os.path.join(TUNE, f"seed{SEED}")
if not os.path.isdir(SEEDDIR):                             # e.g. statefate, whose d20_tune/ does not exist yet
    print(f"  [seed] no {SEEDDIR} -- falling back to the main d{DIM} run")
    SEEDDIR, SEED = RESULTS, "main"
OURS    = "ours: UOT maps"                                 # the method the paper reports (and §2 browses)
# y-tick labels for the metric bars: the panel titles are too long for a narrow column, and a
# rotated or wrapped label is the first casualty at print size.
SHORT   = {"ours: UOT maps": "UOTReg", "ours: OT-CFM": "UOTReg (CFM)", "WOT": "WOT",
           "MMFM": "MMFM", "MioFlow": "MioFlow", "TIGON": "TIGON"}
PRETTY_PANEL = dict(SHORT)   # the same rename for the panel titles (internal keys stay as they are)

arrays, timepoints, _ = A.load_data(DATASET, DIM)
series, TRAJ_T = A.analysis_series(DATASET, DIM)
XLIM, YLIM = A.pc_extent(series)                           # view box from the data (a few strays go off-frame)

RESULTS_ROOT = TP.results_root(shipped=SHIPPED)
_meta_fp = os.path.join(RESULTS, f"traj_meta_{DATASET}{DIM}.json")
_grids = json.load(open(_meta_fp)).get("grids", {}) if os.path.exists(_meta_fp) else {}


def _grid_for(tag, src, tr):
    """The method's own time grid: its folder meta, then the cell's `traj_meta`, then by length."""
    mf = os.path.join(src, f"meta_{DATASET}{DIM}_{tag}.json")
    if os.path.exists(mf):
        return [float(t) for t in json.load(open(mf))["grid"]]
    if tag in _grids:
        return [float(t) for t in _grids[tag]]
    if tr.shape[0] == len(TRAJ_T):
        return [float(t) for t in TRAJ_T]
    if tr.shape[0] == len(timepoints):
        return [float(t) for t in timepoints]
    return list(range(tr.shape[0]))


TRAJ_SRC = {}                   # display name -> the .npy actually loaded (filled by _load_trajs)
TRAJ_IDX = {}                   # display name -> GLOBAL day-0 row id of each trajectory row
#
# A ROW IS NOT A CELL: each method's rows are whatever cell set its run started from (statefate's
# start ids are scattered, not a prefix). Everything below indexes by GLOBAL day-0 id and converts
# per method with `_row`, so a panel never draws a different cell than its neighbour.


def _load_trajs():
    """{display name: (traj, grid) or None}. `None` = that method's run is PENDING.

    Ours comes from the chosen seed folder; everything else resolves through `_traj_paths`, which
    reads MMFM / MioFlow / TIGON ONLY from their own folder. That is deliberate: the flat
    `traj_*_MMFM.npy` files were produced by `trajectory_analysis`'s in-file MMFM at different
    settings, so a fallback would quietly put the wrong fit in the figure under the right name."""
    out = {}
    TRAJ_SRC.clear()
    for name in FIGURE_METHODS:
        tag = TP.TAG.get(name, name)
        if name.startswith("ours") and SEEDDIR != RESULTS:      # the tuned seed wins for ours
            src, fp = SEEDDIR, os.path.join(SEEDDIR, TP.fname(DATASET, DIM, tag))
        else:
            fp = TP.traj_path(RESULTS_ROOT, DATASET, DIM, tag)
            src = os.path.dirname(fp)
        if not os.path.exists(fp):
            out[name] = None
            print(f"  {name:16s} {'-- run pending --':18s} ({os.path.relpath(fp, RESULTS_ROOT)})")
            continue
        tr = np.load(fp)
        grid = _grid_for(tag, src, tr)
        out[name] = (tr, grid)
        TRAJ_SRC[name] = fp            # the metrics cache stamps on this file's mtime
        sif = os.path.join(src, f"startidx_{DATASET}{DIM}_{tag}.npy")
        TRAJ_IDX[name] = (np.load(sif)[:tr.shape[1]].astype(int) if os.path.exists(sif)
                          else np.arange(tr.shape[1], dtype=int))
        print(f"  {name:16s} {str(tr.shape):18s} grid={grid}   [{os.path.basename(src)}]")
    return out


print(f"{DATASET} d={DIM} | ours from: {os.path.relpath(SEEDDIR, REPO)}")
print(f"  observed times {timepoints} | dense grid {TRAJ_T}")
TRAJS = _load_trajs()


def start_point_check(tol=1e-4, verbose=True):
    """Do all methods start from the SAME day-0 coordinates? Report, never silently assume.

    Every method transports the same day-0 cells in the same row order, so `traj[0]` should be the
    observed cloud itself. MioFlow is the documented exception: with `use_gae=True` it encodes into
    the 10-d GAE latent, integrates there and DECODES back (`mioflow_traj.py`, the read-out loop), so
    even its t=0 frame is `decoder(encoder(X0))` -- a reconstruction, not the identity. That is a
    property of the method as its authors run it, not a bug here, and it is why its paths start a
    short distance off the shared start. The CELLS still correspond one-to-one by row, so the
    comparison remains cell-matched; only the coordinates are reconstructed."""
    X0 = np.asarray(arrays[0], np.float32)
    rows = {}
    for m, got in TRAJS.items():
        if not got:
            continue
        t0 = np.asarray(got[0][0], np.float32)
        # index the observed cloud by the run's OWN global ids, not positionally: a tuned statefate
        # run starts from the scattered published 3000, and comparing it row-for-row against the
        # day-0 cloud reported a spurious 22-unit offset for OUR OWN method
        gid = TRAJ_IDX.get(m)
        ref = X0[gid[:len(t0)]] if gid is not None else X0[:len(t0)]
        n = min(len(t0), len(ref))
        rows[m] = float(np.abs(t0[:n] - ref[:n]).max())
    if verbose:
        off = {m: v for m, v in rows.items() if v > tol}
        if off:
            print("  start-point check: " + ", ".join(f"{m} off by {v:.3g}" for m, v in off.items())
                  + "  (MioFlow: expected -- GAE encode/decode round-trip, see start_point_check.__doc__)")
        else:
            print("  start-point check: every method starts from the observed day-0 cells")
    return rows


START_OFFSET = start_point_check()
assert TRAJS.get(OURS), f"no {OURS} trajectory under {SEEDDIR}"
HAVE = [m for m in FIGURE_METHODS if TRAJS.get(m)]
PENDING = [m for m in FIGURE_METHODS if not TRAJS.get(m)]
N_CELLS = TRAJS[OURS][0].shape[1]
# Every method transports every day-0 cell, so this should equal `N_CELLS` -- checked, not assumed.
_ROWMAP = {m: {int(g): i for i, g in enumerate(TRAJ_IDX[m])} for m in HAVE}


def _row(method, gid):
    """Row of GLOBAL cell `gid` in `method`'s trajectory, or None if that run never carried it."""
    return _ROWMAP[method].get(int(gid))


def common_cell_ids():
    """Global day-0 ids every loaded method carries -- the only cells a comparison may draw."""
    return np.array(sorted(set.intersection(*[set(TRAJ_IDX[m].tolist()) for m in HAVE])))


CELL_IDS = common_cell_ids()
N_COMMON = len(CELL_IDS)
print(f"  {N_CELLS} cells in {OURS}; {N_COMMON} carried by EVERY loaded method")
if N_COMMON < N_CELLS:
    print(f"  note: {N_CELLS - N_COMMON} of ours' cells are missing from at least one other method; "
          f"the drawn set comes from the shared ids only.")
if not np.array_equal(TRAJ_IDX[OURS], np.arange(N_CELLS)):
    print(f"  note: {OURS} does NOT start from cells 0..{N_CELLS-1} -- its ids run "
          f"{TRAJ_IDX[OURS][:3].tolist()}... (statefate's 3000 start cells); every method is "
          f"indexed by GLOBAL id here.")

## Shared colour + drawing helpers
Time is mapped through ONE `CMAP_NAME` normalisation over the whole span, so a waypoint's colour
means the same day in every panel even where the methods live on different grids (ours 9 times,
WOT / MMFM 5).

In [ ]:
_ALL_T = sorted({float(t) for v in TRAJS.values() if v for t in v[1]} | {float(t) for t in TRAJ_T})
_NORM  = Normalize(vmin=min(_ALL_T), vmax=max(_ALL_T))
_CMAP  = plt.get_cmap(CMAP_NAME)


def tcolor(t):
    lo, hi = CMAP_RANGE
    return _CMAP(lo + float(_NORM(float(t))) * (hi - lo))


def _sub(X, n, seed=BG_SEED):
    """`n` cells of `X` without replacement (reproducible); all of them if `n` >= len(X)."""
    X = np.asarray(X)
    if n is None or n >= len(X):
        return X
    return X[np.random.default_rng(seed).choice(len(X), n, replace=False)]


def _bg_clouds():
    """(clouds, times) for the §2 background: the raw observed snapshots, or the dense estimated series."""
    if BG_SOURCE == "observed":
        return [arrays[timepoints.index(t)] for t in timepoints], list(timepoints)
    return series, list(TRAJ_T)


def bg_handle(n=None, style=None):
    """The legend entry for the background, worded to match what is actually drawn.

    It says "observed cells" only when the background IS the observed snapshots -- with
    `BG_SOURCE="series"` the interior clouds are estimated, and claiming otherwise in the legend
    would be wrong."""
    st = _style(style)
    n = st["bg_n"] if n is None else n
    clouds, ts = _bg_clouds()
    total = sum(len(c) for c in clouds)
    drawn = sum(min(n, len(c)) for c in clouds) if n else total
    what = "observed cells" if BG_SOURCE == "observed" else "observed + estimated cells"
    label = f"{what} ({drawn}/{total})" if drawn < total else what
    # the swatch must match what was actually drawn: grey in the "dots" style, a mid-time colour in
    # "arrows" (where the background carries the time information and IS coloured)
    mfc = BG_GREY if st["bg_mode"] == "grey" else tcolor(ts[len(ts) // 2])
    return Line2D([0], [0], marker="o", ls="", mec="none", mfc=mfc, alpha=max(st["bg_alpha"], 0.5),
                  label=label)


def draw_background(ax, n=None, mode=None, alpha=None, style=None):
    """Scatter the background clouds. `mode='grey'` = one light grey for everything (the paths read
    best against it); `mode='time'` = the tune file's RdBu_r-by-time look. Defaults come from the
    active STYLE, so a style switch moves colour, alpha and count together."""
    st = _style(style)
    n = (st["bg_n"] if n is None else n)
    mode = (st["bg_mode"] if mode is None else mode)
    alpha = (st["bg_alpha"] if alpha is None else alpha)
    clouds, ts = _bg_clouds()
    for i, (X, t) in enumerate(zip(clouds, ts)):
        P = A.project2d(_sub(X, n))
        c = BG_GREY if mode == "grey" else tcolor(t)
        ax.scatter(P[:, 0], P[:, 1], color=[c], s=BG_S, alpha=alpha, edgecolors="none", zorder=i + 1)
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
    ax.set_xlim(*XLIM); ax.set_ylim(*YLIM)


def arrow_segments(n_seg, k):
    """Which segment indices carry an arrowhead: `k` spread along the path, the LAST always included.

    The last segment is forced because that is where the cell ENDS -- an arrowhead there is what
    makes a path read as directed rather than as an undirected squiggle. With `k=2` the other head
    lands on the first segment, so a path announces its direction where it starts and where it
    stops, and stays a plain line in between."""
    if n_seg <= 0 or k <= 0:
        return []
    k = min(int(k), n_seg)
    if k == 1:
        return [n_seg - 1]
    idx = set(int(j) for j in np.linspace(0, n_seg - 1, k).round().astype(int))
    idx.add(n_seg - 1)
    return sorted(idx)


def draw_path(ax, traj, grid, i, lw=None, arrows=True, nodes=None, node_scale=None, style=None,
              n_arrows=None, halo=None):
    """One cell's path: a black polyline with a FEW arrowheads, optionally with time-coloured waypoints.

    `nodes`: "full" (the tune file's size) | "small" (scaled down, for the grey-background style) |
    "none" (line + arrowheads only -- the WOT / MioFlow convention, where the path carries direction
    and the background carries time).
    `n_arrows` / `halo` override the active style's values for one call."""
    st = _style(style)
    nodes = st["nodes"] if nodes is None else nodes
    node_scale = st["node_scale"] if node_scale is None else node_scale
    n_arrows = (st.get("n_arrows", PATH_ARROWS) if n_arrows is None else n_arrows)
    halo = (st.get("halo", PATH_HALO) if halo is None else halo)
    P = A.project2d(traj[:, i, :])
    lw = PATH_LW if lw is None else lw

    # ONE polyline for the whole path rather than one arrow patch per segment. The shape is carried
    # by the line; only `n_arrows` segments get a head. `withStroke` paints a white casing UNDER the
    # line in the same draw call, so where two paths cross the upper one stays continuous instead of
    # merging into a blob -- and it costs nothing where nothing overlaps.
    pe = [path_effects.withStroke(linewidth=lw + 2 * halo, foreground="white")] if halo else None
    ax.plot(P[:, 0], P[:, 1], "-", color="black", lw=lw, alpha=PATH_ALPHA, zorder=11,
            solid_joinstyle="round", solid_capstyle="round", path_effects=pe)
    if arrows:
        for j in arrow_segments(len(P) - 1, n_arrows):
            # shrinkA/B default to 2pt EACH, which is what left a visible gap at every joint --
            # with them at 0 the head sits flush on the line the polyline already drew.
            ax.add_patch(FancyArrowPatch((P[j, 0], P[j, 1]), (P[j + 1, 0], P[j + 1, 1]),
                                         arrowstyle="-|>", mutation_scale=PATH_MS, lw=lw,
                                         shrinkA=0, shrinkB=0, joinstyle="miter",
                                         color="black", alpha=PATH_ALPHA, zorder=11))
    if nodes == "none":                       # endpoints only -- same marker, the COLOUR says which
        for j in (0, len(P) - 1):
            ax.scatter(P[j, 0], P[j, 1], color=[tcolor(grid[j])], s=ENDNODE_S * 0.55, marker="o",
                       edgecolors="black", linewidths=NODE_EDGE, zorder=12)
        return
    for j, t in enumerate(grid):
        end = j in (0, len(P) - 1)
        sz = (ENDNODE_S if end else NODE_S) * node_scale
        ax.scatter(P[j, 0], P[j, 1], color=[tcolor(t)], s=sz,
                   edgecolors=("black" if end else NODE_EDGE_MID),
                   linewidths=(NODE_EDGE if end else NODE_EDGE * 0.7), zorder=12)


def time_legend(ax_or_fig, times=None, extra=(), **kw):
    """Legend of time swatches (+ any extra handles), placed outside on the right."""
    times = TRAJ_T if times is None else times
    h = [Line2D([0], [0], marker="o", ls="", mfc=tcolor(t), mec="none", label=f"Day {t:g}") for t in times]
    h += list(extra)
    kw.setdefault("loc", "upper left"); kw.setdefault("bbox_to_anchor", (1.01, 1.0))
    kw.setdefault("frameon", False)
    return ax_or_fig.legend(handles=h, **kw)


def legend_block(ax_time, ax_extra=None, times=None, extra=(), ncol_time=2, title="Time",
                 legend_top=1.0):
    """The right column's legend, split across TWO blank axes.

    The day swatches are short and there are up to nine of them, so they tile into `ncol_time`
    columns and take half the height. The "trajectory" / "observed cells (N of M)" entries are long
    single lines that would force those columns as wide as themselves, so they get their own
    one-column legend below.

    Two AXES rather than two legends on one axis: placing the second legend by hand needs the height
    the first one ended up taking, which is not known until the figure is drawn -- the first attempt
    at this overlapped the swatches. Giving each its own gridspec row lets the layout engine do it."""
    times = TRAJ_T if times is None else times
    ax_time.axis("off")
    h = [Line2D([0], [0], marker="o", ls="", mfc=tcolor(t), mec="none", label=f"Day {t:g}")
         for t in times]
    # `ax_extra=None` = there is no LONG entry to isolate, so the short ones ("trajectory") simply
    # join the swatch grid. That is the better arrangement when it applies: a separate legend has to
    # be positioned below the first one, and since both are out of the layout solve, "below" is a
    # guess at how tall the first turned out -- which is what put "trajectory" through "Day 13.5".
    if ax_extra is None and extra is not None:
        h = h + list(extra)
        extra = ()
    leg1 = ax_time.legend(handles=h, ncol=ncol_time, loc="upper left",
                          bbox_to_anchor=(0.0, legend_top),
                          frameon=False, title=title, handletextpad=0.4, columnspacing=0.9,
                          labelspacing=0.35, borderaxespad=0.0)
    leg1._legend_box.align = "left"
    # Keep the legends OUT of the constrained-layout solve. A legend anchored outside its axes is
    # counted as space that axes needs, so the layout engine grew the two legend rows until the bar
    # charts below them were slivers -- the height_ratios were being honoured, then overridden.
    leg1.set_in_layout(False)
    if extra is not None and len(extra) and ax_extra is not None:
        ax_extra.axis("off")
        # a point smaller than the day swatches: these two are supplementary, and "observed cells
        # (1500 of 16821)" is the widest string in the figure -- at legend size it overran the column.
        leg2 = ax_extra.legend(handles=list(extra), ncol=1, loc="upper left",
                               bbox_to_anchor=(0.0, 1.0), frameon=False, handletextpad=0.6,
                               labelspacing=0.5, borderaxespad=0.0, fontsize=9.5 * FS)
        leg2._legend_box.align = "left"
        leg2.set_in_layout(False)
    return ax_time

## 1. Estimated distributions
Every cloud of the anchored series in PC1/PC2, coloured by time. The two **observed** endpoints
are drawn faded and small (context, not the result); the **estimated** interior clouds opaque.
`N_PER_TIME` caps how many cells each time contributes.

In [ ]:
def fig_dist(n_per_time=None, figsize=(9.0, 5.6), title=None, fs=1.0):
    """The series the trajectory is fitted to: two OBSERVED anchors and the estimated clouds between.

    `ends_on_top` keeps the two observed clouds above the interior (statefate's first day is
    otherwise painted over), and `mark_ends` labels them -- the colour scale alone does not show
    that only the endpoints are measured cells."""
    n_per_time = N_PER_TIME if n_per_time is None else n_per_time
    fig, ax = plt.subplots(figsize=figsize, dpi=140)
    A.plot_estimated_distributions_color(
        series, TRAJ_T, ax=ax, max_cells=n_per_time, sub_seed=BG_SEED,
        ends_on_top=True, mark_ends="observed",
        # Title convention shared with the trajectory and gene figures: no colon, no seed,
        # "(d = N)" at the end.
        title=title or f"estimated cell-state distributions in {DATASET} (d = {DIM})")
    ax.title.set_fontsize(12 * fs)
    for lb in (ax.xaxis.label, ax.yaxis.label):
        lb.set_fontsize(10.5 * fs)
    ax.tick_params(labelsize=9.5 * fs)
    if ax.get_legend():
        for t in ax.get_legend().get_texts():
            t.set_fontsize(9 * fs)
    fig.tight_layout()
    return fig


fig_dist()
plt.show()

## 2. Trajectory comparison
The paper draws 20 cells, chosen at random (seed 0) from the day-0 ids that EVERY loaded method
carries, so the same cells appear in every panel.

In [ ]:
KEEP = sorted(int(i) for i in
              np.random.default_rng(0).choice(CELL_IDS, min(N_TRAJ, len(CELL_IDS)), replace=False))
print(f"KEEP ({len(KEEP)} cells, seed 0): {KEEP}")

## The metrics beside the figure
Two numbers per method, both from `A.traj_metrics`: **mean W2** (transported cloud vs observed
snapshot at the observed times) and **purity** (mean aligned-Euclid distance to the k=10 nearest
neighbours' trajectories), both lower = better. `purity_obs` restricts every method to the
observed times first -- `purity_own` would reward grid density, not method quality.

In [ ]:
METRICS_N_SUB = 1000
METRICS_K     = 10
METRICS_SEED  = 0
METRICS_VERSION = 3   # bump when the metric set or its computation changes (invalidates the cache)
METRICS_FP    = os.path.join(RESULTS, f"metrics_bars_{DATASET}{DIM}.json")

# (bar label, key), both lower = better. Purity alone can favour a method that contracts every cell
# onto one path (it is a spread measure), so it is read next to the W2 bar, never alone.
BAR_METRICS = [("mean $W_2$", "fidelity"), ("purity", "purity_obs")]


def _traj_stamp(method):
    """Identity of what a method's number was computed FROM: the file loaded, its mtime, the settings.

    `TRAJ_SRC` is used rather than re-deriving the path, because "ours" resolves through the SEED
    folder while the baselines resolve through `_traj_paths` -- re-deriving would stamp the wrong
    file for ours. Keyed this way, a refit of ONE method invalidates only its own cached row. (The
    stale-cache bug in `seed_analysis` came from keying on the name alone.)"""
    fp = TRAJ_SRC.get(method)
    mt = os.path.getmtime(fp) if fp and os.path.exists(fp) else None
    return {"path": os.path.relpath(fp, REPO) if mt else None, "mtime": mt,
            "n_sub": METRICS_N_SUB, "k": METRICS_K, "seed": METRICS_SEED, "v": METRICS_VERSION}


def metrics_table(methods=None, recompute=False, verbose=True):
    """{method: A.traj_metrics(...)} for every method that has a trajectory, cached on disk.

    A method whose run has not landed is simply absent from the returned dict -- the caller draws it
    as a gap rather than as a zero, because a missing bar and a bar of height zero mean very
    different things."""
    methods = list(methods or FIGURE_METHODS)
    cache = {}
    if os.path.exists(METRICS_FP) and not recompute:
        try:
            cache = json.load(open(METRICS_FP))
        except Exception:
            cache = {}
    out, changed = {}, False
    for m in methods:
        got = TRAJS.get(m)
        if not got:
            continue
        stamp = _traj_stamp(m)
        hit = cache.get(m)
        if hit and hit.get("stamp") == stamp and not recompute:
            out[m] = hit["metrics"]
            continue
        tr, grid = got
        if verbose:
            print(f"  computing metrics for {m} ...", flush=True)
        out[m] = A.traj_metrics(tr, grid, arrays, timepoints, n_sub=METRICS_N_SUB,
                                k=METRICS_K, seed=METRICS_SEED)
        # `spread_err` = |1 - std_pred/std_obs| at the final time, lower better. Not in the bars,
        # but it is what makes purity readable: a low purity beside a large spread_err is a method
        # contracting every cell onto one path, not a coherent one.
        sp_pred = float(np.asarray(tr[-1]).std(0).mean())
        sp_obs = float(np.asarray(arrays[-1]).std(0).mean())
        out[m]["spread_ratio"] = sp_pred / sp_obs if sp_obs else float("nan")
        out[m]["spread_err"] = abs(1.0 - out[m]["spread_ratio"])
        cache[m] = {"stamp": stamp, "metrics": out[m]}
        changed = True
    return out


def metric_bars(ax, M, methods, key, label, highlight=OURS):
    """One metric as a compact horizontal bar chart, method order matching the panels above.

    The drawing itself is `A.metric_barh`, shared with `cross_dimension.py` so the two paper figures
    keep one visual language; this wrapper only pulls the values out of the metrics table."""
    return A.metric_barh(ax, [SHORT.get(m, m) for m in methods],
                         [M.get(m, {}).get(key, float("nan")) for m in methods],
                         label, highlight=SHORT.get(highlight, highlight), fs=FS, arrow=None)

## The figure -- one panel per method
Every panel draws the same cells on the same axes box. Methods read out on the coarser observed
grid (WOT / MMFM) simply have fewer waypoints -- the colour still says which day each one is. The
right-hand column carries the legend and the two metric bar charts.

In [ ]:
# The right column (legend + metric bars) is sized in INCHES, not as a share of the figure: it
# holds fixed-size text that does not shrink as method panels are added. Its rows are ONE flat
# grid -- nesting a sub-subgridspec for the two bar charts defeats constrained_layout's sizing and
# flattens them to slivers.
RIGHT_W       = 3.80                               # column width in inches (at FS 1.30)
RIGHT_ROWS    = (0.42, 0.29, 0.29)                 # legend | mean W2 bars | purity bars
RIGHT_ROWS_BG = (0.355, 0.075, 0.285, 0.285)       # + a row for the long entry (bg_legend=True)
RIGHT_HSPACE  = 0.15
BAR_INSET     = 0.05    # fraction of the column left empty left of the bars (the legend stays full width)
LEGEND_DROP   = 0.06    # how far the legend sits inside its own axes, as a fraction of its height


def fig_compare(cells=None, methods=None, panel=(4.6, 5.15), n_bg=None, mode=None,
                suptitle=None, skip_pending=None, style=None,
                n_traj=None, metrics=True, M=None, bg_legend=False, title_top=0.955):
    """The paper figure: one panel per method, plus a right column of legend + metric bars.

    A method with no saved run keeps an empty panel marked "run pending" (`skip_pending=True` drops
    it instead). `n_traj` thins the selection for this figure only; `metrics=False` drops the bar
    charts and gives the legend the whole column."""
    cells = KEEP if cells is None else list(cells)
    assert cells, "KEEP is empty"
    if n_traj is not None and n_traj < len(cells):
        cells = cells[:int(n_traj)]
    methods = list(methods or FIGURE_METHODS)
    if SKIP_PENDING if skip_pending is None else skip_pending:
        methods = [m for m in methods if TRAJS.get(m)]
    ncols = len(methods)

    # gridspec, not `plt.subplots`: the right column is fixed-width content, so it must not scale
    # with the number of method panels. `layout="constrained"`, not `tight_layout`: that column is a
    # nested gridspec, which tight_layout cannot measure.
    # Panel height 5.15 is settled: printed height = 6.5 * h / w, and w = 4.6*ncols + RIGHT_W. The
    # WIDTH sets printed text size, so widening shrinks every label -- raise FS instead.
    fig = plt.figure(figsize=(panel[0] * ncols + RIGHT_W, panel[1]), dpi=150, layout="constrained")
    # Air under the suptitle, via the layout engine's `rect` -- the axes are confined below
    # `title_top` and the title keeps the strip above it. NOT `h_pad`: that pads EVERY axes, and the
    # right column here is a nested gridspec (legend + two bar charts), so the padding comes out of
    # each of its rows and squashes the bars into overlapping slivers. `tight_layout(rect=...)` is
    # not available at all, because tight_layout cannot measure a nested gridspec.
    fig.get_layout_engine().set(rect=(0, 0, 1, title_top))
    gs = fig.add_gridspec(1, ncols + 1, width_ratios=[1] * ncols + [RIGHT_W / panel[0]])
    axes = []
    for j in range(ncols):
        ax = fig.add_subplot(gs[0, j], sharex=axes[0] if axes else None,
                             sharey=axes[0] if axes else None)
        axes.append(ax)

    short = {}
    for ax, m in zip(axes, methods):
        draw_background(ax, n=n_bg, mode=mode, style=style)
        got = TRAJS.get(m)
        if got is None:                                    # run pending -- background only, said plainly
            ax.text(0.5, 0.5, "run pending", transform=ax.transAxes, ha="center", va="center",
                    fontsize=11 * FS, color="0.35",
                    bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.9))
            ax.set_title(f"{PRETTY_PANEL.get(m, m)}  (pending)", color="0.35")
            continue
        tr, grid = got
        # `cells` are GLOBAL day-0 ids; each method's row for a given cell is looked up, never
        # assumed equal to the id (see the note on TRAJ_IDX)
        rows_here = [(int(i), _row(m, i)) for i in cells]
        drawn = [r for _i, r in rows_here if r is not None]
        missed = [i for i, r in rows_here if r is None]
        if missed:
            short[m] = (tr.shape[1], missed)
        for r in drawn:
            draw_path(ax, tr, grid, r, style=style)
        ax.set_title(PRETTY_PANEL.get(m, m))
    for ax in axes:
        ax.set_xlabel("PC 1")
    axes[0].set_ylabel("PC 2")
    for ax in axes[1:]:                       # shared y: keep one label and one set of ticks
        ax.tick_params(labelleft=False)

    # ---- right column: legend on top, the metric bars underneath ---------------------------------
    # The background swatch is off by default: it is the longest string in the figure, it forced the
    # right column wider than the bars need, and what it says ("observed cells (1500/16821)") belongs
    # in the caption. `bg_legend=True` puts it back; `bg_handle` is unchanged and still used by
    # `show_keep`.
    extra = [Line2D([0], [0], color="black", lw=1.6, label="trajectory")]
    if bg_legend:
        extra.append(bg_handle(n_bg, style))
    if metrics:
        rows = list(RIGHT_ROWS_BG if bg_legend else RIGHT_ROWS)
        rgs = gs[0, ncols].subgridspec(len(rows), 1, height_ratios=rows, hspace=RIGHT_HSPACE)
        n_leg = len(rows) - len(BAR_METRICS)                 # 1 or 2 legend rows
        # The legend is lowered INSIDE its own axes (`legend_top`), not by inserting a blank row: an
        # extra row also adds an extra `hspace` gap, and that came out of the two bar charts -- their
        # method labels started overlapping.
        legend_block(fig.add_subplot(rgs[0]),
                     fig.add_subplot(rgs[1]) if n_leg == 2 else None,
                     TRAJ_T, extra=extra, ncol_time=2, legend_top=1.0 - LEGEND_DROP)
        M = metrics_table(methods) if M is None else M
        for r, (label, key) in enumerate(BAR_METRICS, start=n_leg):
            # Inset each bar chart from the LEFT of its column so it does not butt against the last
            # trajectory panel. A nested 1x2 gridspec with an empty first cell is how this is done
            # under constrained layout -- the axes position cannot simply be set. The legend row is
            # left full-width: it is text, and indenting it would misalign it with the bar labels.
            bgs = rgs[r].subgridspec(1, 2, width_ratios=[BAR_INSET, 1 - BAR_INSET], wspace=0)
            metric_bars(fig.add_subplot(bgs[0, 1]), M, methods, key, label)
    else:
        if bg_legend:
            rgs = gs[0, ncols].subgridspec(2, 1, height_ratios=[0.62, 0.38], hspace=0.3)
            legend_block(fig.add_subplot(rgs[0]), fig.add_subplot(rgs[1]), TRAJ_T, extra=extra,
                         ncol_time=2)
        else:
            legend_block(fig.add_subplot(gs[0, ncols]), None, TRAJ_T, extra=extra, ncol_time=2)

    fig.suptitle(suptitle or f"inferred trajectories in {DATASET} (d = {DIM})")
    for m, (n, missed) in short.items():
        print(f"  ** {m} carries {n} cells and does not include {len(missed)} of the picked ones: "
              f"{missed[:8]}{' ...' if len(missed) > 8 else ''}")
    pend = [m for m in methods if not TRAJS.get(m)]
    if pend:
        print(f"  panels left empty (run pending): {', '.join(pend)}")
    return fig


fig_compare()

## 3. The fate clustering
The same trajectories coloured by cluster. The labels are the settled clustering of the day-0
cells (shipped with the repository; `fates_and_markers` derives the fate call of each cluster).

In [ ]:
LAB_PUB = np.load(P.find("figures", "realdata", "trajs_comparisons",
                         f"labels_published_{DATASET}{DIM}_seed{SEED}.npy"))
ALL_CL = sorted(int(c) for c in np.unique(LAB_PUB))
print(f"{len(LAB_PUB)} day-0 cells in {len(ALL_CL)} clusters | sizes "
      + "  ".join(f"{c}:{int((LAB_PUB == c).sum())}" for c in ALL_CL))


def fig_fates(method=None, per_cluster_max=80, seed=0, title=None):
    """The 1x3 PC-pair panel with each cluster's MEDOID overplotted thick (`A.plot_clusters_medoids`).

    Cluster IDS ONLY in the legend -- deliberately not the fate names: the figure's job is to show
    that the trajectories separate into groups; naming those groups is what the DE tables in
    `fates_and_markers` earn."""
    method = method or OURS
    tr, _grid = TRAJS[method]
    n = min(len(LAB_PUB), tr.shape[1])
    DM = A.pairwise_aligned_euclid(tr[:, :n, :])
    names = {c: f"cluster {c}" for c in ALL_CL}
    fig = A.plot_clusters_medoids(tr[:, :n, :], LAB_PUB[:n], DM, per_cluster_max=per_cluster_max,
                                  seed=seed, names=names,
                                  title=title or f"{DATASET} d={DIM} : trajectories by cluster")
    return fig

In [ ]:
fig_fates()
plt.show()

## 4. Gene dynamics
One panel per gene, clusters overlaid: how each fate's marker moves along the estimated grid.
Expression is reconstructed as `traj @ W.T + mean` through the leading `DIM` PCs.

⚠ Needs the dataset's `.h5ad` (gene space). Embryoid's ships with the repository; statefate's
does not -- see the README.

In [ ]:
import pandas as pd

FS_G = 1.15                        # this section's own text multiplier (narrower panels)
FS = FS_G
N_SPAGHETTI = 12                   # sampled per-cell lines behind the mean
plt.rcParams.update({"font.size": 10 * FS, "axes.titlesize": 11 * FS, "axes.labelsize": 10 * FS,
                     "xtick.labelsize": 9 * FS, "ytick.labelsize": 9 * FS,
                     "legend.fontsize": 9 * FS, "figure.titlesize": 12 * FS})

# The genes on show: ONE per fate cluster, each tied to the cluster `_fate_sets` defines its
# markers at. (Embryoid's sixth cluster is the proliferative/cell-cycle one -- not a lineage, so
# no marker panel.)
GENES = {
    "embryoid": [
        ("BMP4",    1, "ME-like: cardiac",        "prior",  "GOLD in _fate_sets"),
        ("COL1A1",  5, "ME-like: ECM-rich",       "prior",  "the old R script's own example"),
        ("SOX2",    4, "NE-like: central",        "prior",  "retained here, lost in every other cluster"),
        ("TFAP2B",  2, "NE-like: neural crest",   "prior",  "strongest crest call in _fate_sets"),
        ("KRT18",   0, "EN-like: extraembryonic", "prior",  "retained here while lost elsewhere; R2 0.80"),
    ],
    "statefate": [
        ("Cebpe",   0, "Neutrophil",              "prior",  "promyelocyte->myelocyte transition"),
        ("Cd34",    1, "MPP / early progenitor",  "prior",  "declines least here; R2 0.47"),
        ("Cpa3",    2, "Baso/mast arm",           "prior",  "R2 0.84, the cleanest marker in either dataset"),
        ("Ctsl",    3, "Monocyte",                "prior",  "DE #11; R2 0.22 -- Mafb is 0.09, unusable"),
    ],
}[DATASET]
print(f"[gene_dynamics] {DATASET} d={DIM} seed{SEED} | "
      f"{len(GENES)} genes over {len(set(c for _g, c, _f, _s, _w in GENES))} clusters")

In [ ]:
import anndata as ad                                                # noqa: E402

# This section works in gene space, so it needs the dataset's .h5ad. Embryoid's ships with the
# repository; statefate's (~860 MB) does not -- see the README. Say so plainly rather than failing
# inside h5py several cells later.
_H5 = P.data({"embryoid": "embryoid/embryoid_data.h5ad",
              "statefate": "scrna-statefate/invitro-hvg.h5ad"}[DATASET])
if not os.path.exists(_H5):
    raise FileNotFoundError(
        f"{_H5} is not present, so the gene-dynamics section cannot run for {DATASET}. "
        "Every section above works without it; see the README for where to put the file.")

# the seed-of-record run: the tune folder is the SIBLING of the d{DIM} cell
SEEDDIR = os.path.join(os.path.dirname(A.results_dir(DATASET, DIM, shipped=SHIPPED)),
                       f"d{DIM}_tune", f"seed{SEED}")
METHOD = "oursUOTmaps"

lab_pub = LAB_PUB
traj = np.load(os.path.join(SEEDDIR, f"traj_{DATASET}{DIM}_{METHOD}.npy"))
assert len(lab_pub) == traj.shape[1], (len(lab_pub), traj.shape)

H5AD = {"embryoid": "embryoid/embryoid_data.h5ad",
        "statefate": "scrna-statefate/invitro-hvg.h5ad"}[DATASET]
adata = ad.read_h5ad(P.data(H5AD))
X = adata.X.toarray() if hasattr(adata.X, "toarray") else np.asarray(adata.X)
# the " (ENSG...)" suffix has to come off or every symbol lookup misses -- the same trap `de_tables`
# documents, where it silently zeroed the published-cluster overlap
gene_names = np.asarray([str(g).split(" (")[0].strip() for g in adata.var_names])
W = np.asarray(adata.varm["PCs"])[:, :DIM]
gene_means = X.mean(0)
TRAJ_T = A.CFG[DATASET]["traj_t"]
assert traj.shape[0] == len(TRAJ_T), (traj.shape, TRAJ_T)
print(f"  trajectory {traj.shape} on t = {TRAJ_T}")
print(f"  clusters (published ids): " + "  ".join(f"{c}:{int((lab_pub == c).sum())}"
                                                  for c in sorted(set(lab_pub.tolist()))))


UP = np.char.upper(gene_names)          # statefate is MOUSE ('Cpa3'), embryoid HUMAN ('SOX2') --
#                                        an exact-case lookup silently misses every statefate gene
_S = (X - gene_means) @ W               # PC scores of the real cells, for the PC-R2 below


def gene_index(sym):
    hit = np.where(UP == str(sym).upper())[0]
    assert len(hit), f"{sym!r} is not among this dataset's {len(gene_names)} genes"
    return int(hit[0])


def gene_curve(g):
    """(T, N) expression of gene `g` along the trajectory: the PC path mapped back to gene space.

    One column at a time -- the full `traj @ W.T` is (T x N x n_genes) and is not needed for six."""
    return traj @ W[g, :] + gene_means[g]


def pc_r2(g):
    """Fraction of this gene's variance the leading DIM PCs carry. The trajectory lives in that span,
    so this is the ceiling on how much of the gene's real behaviour any curve here could show."""
    rec = _S @ W[g, :]
    v = np.var(X[:, g] - gene_means[g])
    return float(1 - np.var(X[:, g] - gene_means[g] - rec) / v) if v > 0 else float("nan")

In [ ]:
def cell_slopes(curve):
    """Per-cell OLS slope of `curve` (T, N) against the trajectory time grid."""
    d = np.asarray(TRAJ_T, float)
    d = d - d.mean()
    return ((curve - curve.mean(0)) * d[:, None]).sum(0) / float((d ** 2).sum())


def rank_biserial(v, sel):
    """`2*AUC - 1` for `v[sel]` against `v[~sel]`, via the rank-sum identity (no scipy needed)."""
    a, b = np.asarray(v)[sel], np.asarray(v)[~sel]
    n, m = len(a), len(b)
    if n == 0 or m == 0:
        return float("nan")
    r = np.argsort(np.argsort(np.concatenate([a, b]))) + 1.0     # ranks, ties broken arbitrarily
    u = r[:n].sum() - n * (n + 1) / 2.0
    return float(2.0 * (u / (n * m)) - 1.0)

## The figure
One panel per gene; one curve per cluster. The bold line with a band is the gene's own cluster
(mean ± SE along the inferred trajectory), the pale lines are the other clusters, and the thin
grey lines are a sample of individual cells.

In [ ]:
CLUSTERS = sorted(set(int(c) for _g, c, _f, _s, _w in GENES))
ALL_CLUSTERS = sorted(set(int(c) for c in np.unique(lab_pub)))
PALETTE = plt.get_cmap("tab10")
CCOLOR = {c: PALETTE(i % 10) for i, c in enumerate(ALL_CLUSTERS)}
CNAME = {int(c): f for _g, c, f, _s, _w in GENES}


def figure(ncols=None, panel=(3.4, 2.8)):
    ncols = ncols or (3 if len(GENES) > 4 else min(len(GENES), 4))
    nrows = int(np.ceil(len(GENES) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel[0] * ncols, panel[1] * nrows),
                             dpi=150, squeeze=False)
    rows, rng = [], np.random.default_rng(0)
    for ax, (sym, own, fate, src, _why) in zip(axes.ravel(), GENES):
        g = gene_index(sym)
        curve, r2 = gene_curve(g), pc_r2(g)
        for c in ALL_CLUSTERS:                       # EVERY cluster on EVERY panel: specificity is
            sel = lab_pub == c                       # the claim, so the comparison must be visible
            if not sel.any():
                continue
            e = curve[:, sel]
            mu, se = e.mean(1), e.std(1) / np.sqrt(sel.sum())
            is_own, col = (c == own), CCOLOR[c]
            if is_own:                               # cell-level spread, own cluster only
                for j in rng.choice(e.shape[1], min(N_SPAGHETTI, e.shape[1]), replace=False):
                    ax.plot(TRAJ_T, e[:, j], color="0.78", lw=0.4, alpha=0.6, zorder=1)
                ax.fill_between(TRAJ_T, mu - se, mu + se, color=col, alpha=0.20, lw=0, zorder=2)
            ax.plot(TRAJ_T, mu, "-", color=col, lw=2.2 if is_own else 0.9,
                    alpha=1.0 if is_own else 0.45, zorder=4 if is_own else 3,
                    label=(f"cluster {c}" if ax is axes.ravel()[0] else None))
            rows.append(dict(gene=gene_names[g], own_cluster=own, cluster=c, fate=fate,
                             n_cells=int(sel.sum()),
                             change=float(mu[-1] - mu[0]), level_end=float(mu[-1]), own=is_own,
                             pc_r2=r2, src=src,
                             # the two model-free contrasts, computed once per (gene, cluster)
                             rb_slope=rank_biserial(cell_slopes(curve), sel),
                             rb_level=rank_biserial(curve[-1], sel),
                             # the WITHIN-cluster trend, which is a different thing from the
                             # between-cluster contrast above and is what the monotone check needs
                             slope_own=float(np.median(cell_slopes(curve)[sel]))))
        ax.set_title(f"{gene_names[g]}   (cluster {own})\n{fate}", fontsize=9.5 * FS)
        ax.set_xlabel("estimated time")
    for ax in axes.ravel()[:len(GENES)][::ncols]:
        ax.set_ylabel("expression")
    spare = axes.ravel()[len(GENES):]
    for ax in spare:
        ax.axis("off")
    # Five panels in a 3x2 grid leave a hole; put the legend in it rather than inside the first
    # panel, where it sits on top of the curves it is labelling.
    if len(spare):
        # Build the handles rather than lifting them from panel 1: there, the panel's OWN cluster is
        # drawn thick, and a shared legend that inherits that makes one cluster look globally
        # special. Uniform width here; the bold line is a per-panel device, explained in the caption.
        h = [Line2D([0], [0], color=CCOLOR[c], lw=1.6) for c in ALL_CLUSTERS]
        spare[0].legend(h, [f"cluster {c}" for c in ALL_CLUSTERS], fontsize=9 * FS, frameon=False,
                        loc="center", ncol=1, title="trajectory cluster", title_fontsize=9 * FS)
    else:
        axes.ravel()[0].legend(fontsize=7 * FS, frameon=False, loc="best", ncol=2)
    # Title convention shared with the trajectory figures: no colon, no seed, "(d = N)" at the end.
    fig.suptitle(f"marker gene dynamics along the inferred trajectory in {DATASET} (d = {DIM})")
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    return fig, pd.DataFrame(rows)


FIG, TAB = figure()
plt.show()